In [ ]:

# start_time = time.time()
# end_time = time.time()
# execution_time = end_time - start_time
# print("Total execution time:", execution_time, "seconds")

import pandas as pd
import numpy as np
import seaborn as sns
import nltk
import multiprocessing
import swifter
import matplotlib.pyplot as plt
import time
import spacy
import contractions
import string
import re
import os
import fasttext
import gc
import math
# import glove 
start_time = time.time()
# from glove import Corpus, Glove
from sklearn.utils import shuffle
from collections import Counter
from nltk.corpus import stopwords
from IPython.display import Audio
from sklearn.pipeline import Pipeline
from bs4 import BeautifulSoup
from unidecode import unidecode
from transformers import AutoTokenizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix , classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn import preprocessing
from multiprocessing import Pool
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import accuracy_score

#φορτώνουμε τις βιλιοθήκες
gc.collect()

In [ ]:
flag=2

if flag==1:

#     news_df = pd.read_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv')
    
    original_dataset = pd.read_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv', nrows=1000000)
    fake_data = original_dataset[original_dataset['label'] == 'Fake']
    real_data = original_dataset[original_dataset['label'] == 'Real']

    num_fake_samples = 4500
    num_real_samples = 5500

    fake_samples = fake_data.sample(n=num_fake_samples, random_state=56)
    real_samples = real_data.sample(n=num_real_samples, random_state=56)

    news_df = pd.concat([fake_samples, real_samples], ignore_index=True)
    news_df = shuffle(news_df, random_state=56)
    news_df = news_df.reset_index(drop=True)
    
    
    news_df['label'] = "__label__" + news_df['label'].astype(str)
    news_df['label_text'] = news_df['label'] + " " + news_df['text']
    train_df, test_df = train_test_split(news_df, test_size = 0.3, random_state = 56)
    train_df.to_csv('Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_train.txt', columns=['label_text'], index=False, header=False)
    test_df.to_csv('Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_test.txt', columns=['label_text'], index=False, header=False)
    
elif flag==2:
    
    file_path = 'Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_train.txt'
#     model = fasttext.train_supervised(input=file_path, autotuneValidationFile='Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_test.txt' , autotuneDuration=46800 ,loss='hs')
    model = fasttext.train_supervised(input=file_path, dim=300, lr=0.3, epoch=100, thread=12, verbose=1, wordNgrams = 3)
    print(model.test('Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_test.txt'))
    model.save_model('Data Set/Huge dataset/fasttext_model_TextClass_Test3.bin')
    
elif flag==3:
    
    model = fasttext.load_model('Data Set/Huge dataset/fasttext_model_TextClass_Test3.bin')

In [ ]:
# news_df = pd.read_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv')
# # news_df = news_df.head(1000000)

In [ ]:
# news_df['label'].value_counts()

In [ ]:
# #Process the dataset so fasttext can be trained (has to be like this)
# news_df['label'] = "__label__" + news_df['label'].astype(str)
# news_df.head(50)

In [ ]:
# news_df['label_text'] = news_df['label'] + " " + news_df['text']
# news_df.head(50)

In [ ]:
# train_df, test_df = train_test_split(news_df, test_size = 0.3, random_state = 56)

In [ ]:
# train_df.to_csv('Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_train.txt', columns=['label_text'], index=False, header=False)
# test_df.to_csv('Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_test.txt', columns=['label_text'], index=False, header=False)

In [ ]:
# train_df.shape, test_df.shape

In [ ]:
# model = fasttext.load_model('Data Set/Huge dataset/fasttext_model_TextClass.bin')

In [ ]:
# file_path = 'Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_train.txt'
# model = fasttext.train_supervised(input=file_path, dim=300, lr=0.1, epoch=10, thread=10, verbose=1, wordNgrams = 3)

In [ ]:
# model.save_model('Data Set/Huge dataset/fasttext_model_TextClass.bin')

In [ ]:
# model.test('Data Set/Huge dataset/HugeDatasetWhole_LabelsFR_test.txt')

In [ ]:
model.get_nearest_neighbors("trump")

In [ ]:
# model.predict("")

In [ ]:
# liar_dataset.to_csv('Data Set/LIAR dataset/liar_prepro_MERGED_df_nolemmaNstopwords.csv', index=False)

In [ ]:
#NEW: 1 Fake, 0 real
original_dataset = pd.read_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv', nrows=1000000)
fake_data = original_dataset[original_dataset['label'] == 'Fake']
real_data = original_dataset[original_dataset['label'] == 'Real']

num_fake_samples = 45000
num_real_samples = 55000

fake_samples = fake_data.sample(n=num_fake_samples, random_state=56)
real_samples = real_data.sample(n=num_real_samples, random_state=56)

news_df = pd.concat([fake_samples, real_samples], ignore_index=True)
news_df = shuffle(news_df, random_state=56)
news_df = news_df.reset_index(drop=True)

all_labels_huge = news_df['label']

predictions_huge = []
for text in news_df['text']:
    prediction = model.predict(text)
    label = prediction[0][0].replace('__label__', '')
    if label in ['Real', 'Fake']:
        predictions_huge.append(label)

print(classification_report(all_labels_huge, predictions_huge))

In [ ]:
#NEW: 1 Fake, 0 real
welfake_dataset = pd.read_csv('Data Set/WELFake Dataset/new_WELFake_prepro_df_nolemmaNstopwordsNreuters.csv')
welfake_dataset['text'].fillna('', inplace=True)
welfake_dataset['label'] = welfake_dataset['label'].replace({1: 'Fake', 0: 'Real'})

true_labels_welfake = welfake_dataset['label']

predictions_welfake = []
for text in welfake_dataset['text']:
    prediction = model.predict(text)
    label = prediction[0][0].replace('__label__', '')
    if label in ['Real', 'Fake']:
        predictions_welfake.append(label)

print(classification_report(true_labels_welfake, predictions_welfake))

In [ ]:
liar_dataset_test = pd.read_csv('Data Set/LIAR dataset/liar_prepro_test_df_nolemmaNstopwords.csv')
liar_dataset_train = pd.read_csv('Data Set/LIAR dataset/liar_prepro_train_df_nolemmaNstopwords.csv')
liar_dataset_valid = pd.read_csv('Data Set/LIAR dataset/liar_prepro_valid_df_nolemmaNstopwords.csv')
liar_dataset = pd.concat([liar_dataset_test, liar_dataset_train, liar_dataset_valid], ignore_index=True)

true_labels_liar = liar_dataset['label']

predictions_liar = []
for text in liar_dataset['text']:
    prediction = model.predict(text)
    label = prediction[0][0].replace('__label__', '')
    if label in ['Real', 'Fake']:
        predictions_liar.append(label)

print(classification_report(true_labels_liar, predictions_liar))

In [ ]:
liar_dataset_test = pd.read_csv('Data Set/LIAR dataset/liar_prepro_test_df_nolemmaNstopwords.csv')
liar_dataset_train = pd.read_csv('Data Set/LIAR dataset/liar_prepro_train_df_nolemmaNstopwords.csv')
liar_dataset_valid = pd.read_csv('Data Set/LIAR dataset/liar_prepro_valid_df_nolemmaNstopwords.csv')
liar_dataset = pd.concat([liar_dataset_test, liar_dataset_train, liar_dataset_valid], ignore_index=True)

# liar_valid_df = pd.read_csv('Data Set/LIAR dataset/liar_prepro_valid_df_nolemmaNstopwords.csv')
# liar_valid_array_of_strings = liar_valid_df['text'].values.astype(str)
liar_valid_array_of_strings = []

for _, row in liar_dataset.iterrows():
    text = row['text']
    liar_valid_array_of_strings.append(text)

# Print the texts
print(liar_valid_array_of_strings)


In [ ]:
# model.predict(liar_valid_array_of_strings)

In [ ]:
chunk_size = 500
output_file = 'Data set/Huge dataset/LIAR_fasttext_predictions.txt'

with open(output_file, 'w') as f_out:
    for chunk in pd.read_csv('Data Set/LIAR dataset/liar_prepro_MERGED_df_nolemmaNstopwords.csv', chunksize=chunk_size):
        for index, row in chunk.iterrows():
            text = str(row['text'])

            prediction = model.predict(text)

            label = prediction[0][0].replace('__label__', '')
            if label in ['Real', 'Fake']:
                f_out.write(f"{label}\n")

In [ ]:
liar_pred_path = 'Data set/Huge dataset/LIAR_fasttext_predictions.txt'
with open(liar_pred_path, 'r') as file:
    liar_predicted_labels = [line.strip() for line in file.readlines()]

print(liar_predicted_labels)

In [ ]:
# liar_predicted_labels = [
#     ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Real'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake'],   ['__label__Fake']
# ]

In [ ]:
# liar_cleaned_labels = []
# for label in liar_predicted_labels:
#     if label[0] == '__label__Fake':
#         liar_cleaned_labels.append('Fake')
#     elif label[0] == '__label__Real':
#         liar_cleaned_labels.append('Real')
        
# print(liar_cleaned_labels)

In [ ]:
title_original = 'Original labels'
title_fasttext = 'fasttext labels'

print(f"{title_original:<15}\t\t{title_fasttext}")


for value_og, value_fasttext in zip(liar_dataset['label'], liar_predicted_labels):
    print(f"\t{value_og:<15}\t\t{value_fasttext}")

In [ ]:
#Calculate the correctness of the prediction in comparision of the input_testing_df['labels']

actual_labels = liar_dataset['label']

correct_predictions = sum(liar_predicted_labels == actual_labels)

total_predictions = len(liar_predicted_labels)

correctness_percentage = (correct_predictions / total_predictions) * 100

print(f"Correctness: {correctness_percentage:.2f}%")

In [ ]:
model.get_nearest_neighbors("poverty")

In [ ]:
chunk_size = 500
output_file = 'Data set/Huge Dataset/WELFake_fasttext_predictions2.txt'

with open(output_file, 'w') as f_out:
    for chunk in pd.read_csv('Data Set/WELFake Dataset/new_WELFake_prepro_df_nolemmaNstopwordsNreuters.csv', chunksize=chunk_size):
        for index, row in chunk.iterrows():
            text = str(row['text'])

            prediction = model.predict(text)

            label = prediction[0][0].replace('__label__', '')
            if label in ['Real', 'Fake']:
                f_out.write(f"{label}\n")

In [ ]:
# WELFake_valid_df = pd.read_csv('Data Set/WELFake Dataset/new_WELFake_prepro_df_nolemmaNstopwords.csv', dtype=str, nrows=500)
# WELFake_valid_array_of_strings = []

# for _, row in WELFake_valid_df.iterrows():
#     text = row['text']
#     WELFake_valid_array_of_strings.append(text)


# type(WELFake_valid_array_of_strings)
# print(WELFake_valid_array_of_strings)

In [ ]:
# WELFake_valid_df

In [ ]:
# model.predict(WELFake_valid_array_of_strings)

In [ ]:
WELFake_pred_path = 'Data set/Huge Dataset/WELFake_fasttext_predictions2.txt'
with open(WELFake_pred_path, 'r') as file:
    WELFake_predicted_labels = [line.strip() for line in file.readlines()]

print(WELFake_predicted_labels)

In [ ]:
#NEW: 1 Fake, 0 real
WELFake_valid_df = pd.read_csv('Data Set/WELFake Dataset/new_WELFake_prepro_df_nolemmaNstopwordsNreuters.csv')
label_mapping = {1: 'Fake', 0: 'Real'}
WELFake_valid_df['label'] = WELFake_valid_df['label'].replace(label_mapping)

title_original = 'Original labels'
title_fasttext = 'fasttext labels'

print(f"{title_original:<15}\t\t{title_fasttext}")


for value_og, value_fasttext in zip(WELFake_valid_df['label'], WELFake_predicted_labels):
    print(f"\t{value_og:<15}\t\t{value_fasttext}")

In [ ]:
#Calculate the correctness of the prediction in comparision of the input_testing_df['labels']

actual_labels = WELFake_valid_df['label']

correct_predictions = sum(WELFake_predicted_labels == actual_labels)

total_predictions = len(WELFake_predicted_labels)

correctness_percentage = (correct_predictions / total_predictions) * 100

print(f"Correctness: {correctness_percentage:.2f}%")

In [ ]:
# drop_values = ['political', 'bias', 'conspiracy', 'rumor', 'unreliable', 'clickbait', 'junksci', 'satire', 'hate']
# news_df = news_df.drop(news_df[news_df['label'].isin(drop_values)].index)
# news_df = news_df.reset_index(drop=True)

# value_mapping = {'fake': 'Fake', 'reliable': 'Real'}
# news_df['label'] = news_df['label'].replace(value_mapping)

# news_df.to_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv', index=False)

# news_df['label'].value_counts()

# news_df

In [ ]:
end_time = time.time()
execution_time = end_time - start_time
print("Total execution time:", execution_time, "seconds")
Audio(filename=r'C:\Users\Poustols\Downloads\hapi-hapi-hapi.mp3', autoplay=True)